# AIDatasetKit vs. a Conventional Expert Workflow
## Real-world, reproducible comparative study

**Purpose.** Compare two legitimate approaches on published public datasets:

1. **Conventional expert workflow** — explicit `pandas` + `scikit-learn` pipelines built carefully by hand.
2. **AIDatasetKit workflow** — the high-level facade and the library's safety/evidence contracts.

The comparison is intentionally **not** "good library vs bad manual code". The manual baseline should be competent and leakage-aware.

### Research questions
- **RQ1 — Parity:** Do both approaches produce scientifically comparable results under matched conditions?
- **RQ2 — Safety:** Which workflow better detects/prevents leakage, ID contamination, invalid numerics, and task/target mismatches?
- **RQ3 — Transparency:** Which workflow makes preprocessing, lineage, and evidence easier to inspect?
- **RQ4 — Efficiency:** How much orchestration code is removed, and what runtime overhead is added?
- **RQ5 — Flexibility:** Where does manual expert control remain superior?
- **RQ6 — Education:** Which workflow better connects concepts to explicit operations without hiding the science?

## Reproducibility metadata and public-data policy

This notebook is interactive: it may download public UCI data through `ucimlrepo`, but no network loading belongs in the package test suite. Each run prints the Python, pandas, NumPy, scikit-learn, and AIDatasetKit versions, seed, validation fraction, dataset IDs, source links, target interpretation, and sampling rule. All source datasets are CC BY 4.0 and must be attributed.[1] [2] [3] [4]

The notebook contains source only by default: it does not embed raw datasets or large fitted models. Re-run it in a network-enabled environment to create local results.

## Dataset sources, interpretation, and ethical limits

| Task | Dataset and source | Notebook interpretation | Caution |
|---|---|---|---|
| Classification / reduction | [Breast Cancer Wisconsin (Diagnostic), UCI 17](https://archive.ics.uci.edu/dataset/17/breast+cancer+wisconsin+diagnostic), DOI [10.24432/C5DW2B](https://doi.org/10.24432/C5DW2B) | `Diagnosis` target; UCI `ID` excluded | Educational benchmark only; not clinical validation or diagnosis. |
| Regression | [Wine Quality, UCI 186](https://archive.ics.uci.edu/dataset/186/wine+quality), DOI [10.24432/C56S3T](https://doi.org/10.24432/C56S3T) | Sensory `quality` deliberately treated as numeric | UCI also supports a classification framing; this is a documented choice. |
| Clustering | [Wholesale customers, UCI 292](https://archive.ics.uci.edu/dataset/292/wholesale+customers), DOI [10.24432/C5030X](https://doi.org/10.24432/C5030X) | Spending features only; UCI `Region` target excluded | Internal scores do not prove semantic customer segments. |
| Anomaly detection | [Statlog (Shuttle), UCI 148](https://archive.ics.uci.edu/dataset/148/statlog+shuttle), DOI [10.24432/C5WS31](https://doi.org/10.24432/C5WS31) | Labels withheld during fitting, then used only as an external benchmark | Minority class is not a universal scientific definition of anomaly. |

All four sources are published through UCI under CC BY 4.0.[5]

## Reproducibility metadata and public-data policy

This notebook is interactive: it may download public UCI data through `ucimlrepo`, but no network loading belongs in the package test suite. Each run prints the Python, pandas, NumPy, scikit-learn, and AIDatasetKit versions, seed, validation fraction, dataset IDs, source links, target interpretation, and sampling rule. All source datasets are CC BY 4.0 and must be attributed.[1] [2] [3] [4]

The notebook contains source only by default: it does not embed raw datasets or large fitted models. Re-run it in a network-enabled environment to create local results.

## Dataset sources, interpretation, and ethical limits

| Task | Dataset and source | Notebook interpretation | Caution |
|---|---|---|---|
| Classification / reduction | [Breast Cancer Wisconsin (Diagnostic), UCI 17](https://archive.ics.uci.edu/dataset/17/breast+cancer+wisconsin+diagnostic), DOI [10.24432/C5DW2B](https://doi.org/10.24432/C5DW2B) | `Diagnosis` target; UCI `ID` excluded | Educational benchmark only; not clinical validation or diagnosis. |
| Regression | [Wine Quality, UCI 186](https://archive.ics.uci.edu/dataset/186/wine+quality), DOI [10.24432/C56S3T](https://doi.org/10.24432/C56S3T) | Sensory `quality` deliberately treated as numeric | UCI also supports a classification framing; this is a documented choice. |
| Clustering | [Wholesale customers, UCI 292](https://archive.ics.uci.edu/dataset/292/wholesale+customers), DOI [10.24432/C5030X](https://doi.org/10.24432/C5030X) | Spending features only; UCI `Region` target excluded | Internal scores do not prove semantic customer segments. |
| Anomaly detection | [Statlog (Shuttle), UCI 148](https://archive.ics.uci.edu/dataset/148/statlog+shuttle), DOI [10.24432/C5WS31](https://doi.org/10.24432/C5WS31) | Labels withheld during fitting, then used only as an external benchmark | Minority class is not a universal scientific definition of anomaly. |

All four sources are published through UCI under CC BY 4.0.[5]

## Public datasets

This notebook uses the **UCI Machine Learning Repository** through `ucimlrepo`.

| Task family | Dataset | UCI ID | Research role |
|---|---|---:|---|
| Classification | Breast Cancer Wisconsin (Diagnostic) | 17 | Binary classification + safety stress tests |
| Regression | Wine Quality | 186 | Regression on sensory quality score |
| Clustering | Wholesale Customers | 292 | Unsupervised customer segmentation |
| Anomaly detection | Statlog (Shuttle) | 148 | Unsupervised fitting; labels used only after fitting as an external benchmark |
| Dimensionality reduction | Breast Cancer Wisconsin (Diagnostic) | 17 | PCA/SVD/ICA diagnostics |

**Important:** for anomaly detection, the Shuttle class labels are not used to fit the detector. Treating minority classes as an anomaly benchmark is an experimental convention, not a real-world causal claim.

In [1]:
# Environment setup
# Run once if needed:
# %pip install ucimlrepo pandas numpy scipy scikit-learn matplotlib

import sys, time, inspect
import numpy as np
import pandas as pd
from ucimlrepo import fetch_ucirepo

import sklearn
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    mean_absolute_error, mean_squared_error, r2_score,
    silhouette_score, davies_bouldin_score, calinski_harabasz_score,
)
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.ensemble import IsolationForest
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("scikit-learn:", sklearn.__version__)

Python: 3.12.3
pandas: 3.0.5
NumPy: 2.5.1
scikit-learn: 1.9.0


In [2]:
from aidatasetkit import AIDataFacade, KitConfig
from aidatasetkit.models import ModelFactory
import aidatasetkit

print("AIDatasetKit:", aidatasetkit.__version__)
print("Registered task families:", [
    "classification", "regression", "clustering", "anomaly_detection", "dimensionality_reduction"
])

AIDatasetKit: 0.1.0a1
Registered task families: ['classification', 'regression', 'clustering', 'anomaly_detection', 'dimensionality_reduction']


In [3]:
def fetch_uci_frame(dataset_id: int):
    ds = fetch_ucirepo(id=dataset_id)
    X = ds.data.features.copy()
    y = None if ds.data.targets is None else ds.data.targets.copy()
    return X, y, ds.metadata, ds.variables

DATASETS = {}
for name, dataset_id in {
    "breast_cancer": 17,
    "wine_quality": 186,
    "wholesale": 292,
    "shuttle": 148,
}.items():
    X, y, meta, variables = fetch_uci_frame(dataset_id)
    DATASETS[name] = {"X": X, "y": y, "meta": meta, "variables": variables}
    print(name, X.shape, None if y is None else y.shape)

breast_cancer (569, 30) (569, 1)


wine_quality (6497, 11) (6497, 1)


wholesale (440, 7) (440, 1)


shuttle (58000, 7) (58000, 1)


# Fair comparison protocol

For each task:
- use the same raw rows whenever possible;
- use the same seed and split policy;
- match model family and important parameters;
- never fit manual preprocessing on evaluation/test rows;
- separate clean-data parity from controlled stress tests;
- record failures instead of deleting them;
- compare end-to-end runtime, not estimator fit alone;
- never declare a winner from predictive metrics alone.

Primary dimensions:
1. task metric/diagnostic,
2. leakage resistance,
3. input validation,
4. preprocessing transparency,
5. lineage/evidence,
6. reproducibility,
7. orchestration burden,
8. runtime,
9. flexibility,
10. error actionability.

In [4]:
RANDOM_STATE = 42
VALIDATION_SIZE = 0.20

def make_config():
    sig = inspect.signature(KitConfig)
    kwargs = {}
    if "random_state" in sig.parameters:
        kwargs["random_state"] = RANDOM_STATE
    if "validation_size" in sig.parameters:
        kwargs["validation_size"] = VALIDATION_SIZE
    return KitConfig(**kwargs)

def elapsed(fn, *args, **kwargs):
    t0 = time.perf_counter()
    value = fn(*args, **kwargs)
    return value, time.perf_counter() - t0

def numeric_categorical_columns(df):
    numeric = list(df.select_dtypes(include=[np.number]).columns)
    categorical = [c for c in df.columns if c not in numeric]
    return numeric, categorical

def manual_preprocessor(X, scale_numeric=True):
    numeric, categorical = numeric_categorical_columns(X)
    num_steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale_numeric:
        num_steps.append(("scaler", StandardScaler()))
    num_pipe = Pipeline(num_steps)
    cat_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])
    return ColumnTransformer([
        ("num", num_pipe, numeric),
        ("cat", cat_pipe, categorical),
    ])

# Study 1 — Classification: Breast Cancer Wisconsin (Diagnostic)

First run a competent manual baseline, then AIDatasetKit. After clean parity, apply controlled safety perturbations.

In [5]:
bc_X = DATASETS["breast_cancer"]["X"].copy()
bc_y = DATASETS["breast_cancer"]["y"].copy()
if isinstance(bc_y, pd.DataFrame):
    bc_y = bc_y.iloc[:, 0]

bc = bc_X.copy()
bc["Diagnosis"] = bc_y.values

X_train, X_valid, y_train, y_valid = train_test_split(
    bc_X, bc_y, test_size=VALIDATION_SIZE,
    random_state=RANDOM_STATE, stratify=bc_y
)

manual_clf = Pipeline([
    ("prep", manual_preprocessor(X_train, scale_numeric=True)),
    ("model", LogisticRegression(max_iter=5000, random_state=RANDOM_STATE)),
])
_, fit_s = elapsed(manual_clf.fit, X_train, y_train)
pred = manual_clf.predict(X_valid)
proba = manual_clf.predict_proba(X_valid)
classes = list(manual_clf.named_steps["model"].classes_)
positive = classes[-1]
pos_idx = classes.index(positive)

manual_classification_metrics = {
    "accuracy": accuracy_score(y_valid, pred),
    "precision_macro": precision_score(y_valid, pred, average="macro", zero_division=0),
    "recall_macro": recall_score(y_valid, pred, average="macro", zero_division=0),
    "f1_macro": f1_score(y_valid, pred, average="macro", zero_division=0),
    "roc_auc": roc_auc_score((y_valid == positive).astype(int), proba[:, pos_idx]),
    "fit_seconds": fit_s,
}
manual_classification_metrics

{'accuracy': 0.9649122807017544,
 'precision_macro': 0.9672297297297296,
 'recall_macro': 0.9573412698412699,
 'f1_macro': 0.9619111259605746,
 'roc_auc': 0.996031746031746,
 'fit_seconds': 0.014273045992013067}

In [6]:
# Matched split parity: positions drawn manually and then compared with the facade comparison split.
manual_train_positions, manual_eval_positions = train_test_split(
    np.arange(len(bc)), test_size=VALIDATION_SIZE, random_state=RANDOM_STATE,
    stratify=bc_y, shuffle=True,
)
manual_train_positions = np.sort(manual_train_positions)
manual_eval_positions = np.sort(manual_eval_positions)
positive_label = "M" if "M" in set(bc_y) else sorted(pd.Series(bc_y).unique())[-1]

ai_clf = AIDataFacade(
    target="Diagnosis", task="classification", positive_label=positive_label, config=make_config()
)
ai_clf.load(bc)
quality = ai_clf.check_quality()
plans = ai_clf.prepare()
comparison = ai_clf.compare_models(models=["logistic_regression", "decision_tree_classifier", "dummy_classifier"])
assert np.array_equal(manual_train_positions, comparison.split.train_positions)
assert np.array_equal(manual_eval_positions, comparison.split.evaluation_positions)

ai_clf.select_model("logistic_regression")
training = ai_clf.train()
evaluation = ai_clf.evaluate()
{
    "split_fingerprint": comparison.split.fingerprint,
    "quality_verdict": ai_clf.verdict.value,
    "quality_issue_count": quality.issue_count,
    "preprocessing_plan_count": len(plans),
    "selected_model": training.model_name,
    "metrics": {name: evaluation[name].value for name in ("accuracy", "precision", "recall", "f1", "roc_auc")},
}

{'split_fingerprint': '8a45f57d7ea299c9',
 'quality_verdict': 'review_required',
 'quality_issue_count': 29,
 'preprocessing_plan_count': 5,
 'selected_model': 'logistic_regression',
 'metrics': {'accuracy': 0.9649122807017544,
  'precision': 0.975,
  'recall': 0.9285714285714286,
  'f1': 0.9512195121951219,
  'roc_auc': 0.996031746031746}}

## Classification safety stress tests

Controlled perturbations of the real dataset:
1. exact target leakage,
2. unique ID feature,
3. infinity,
4. object-dtype numerics,
5. task contradiction.

These are not defects in the original UCI dataset; they are experimental stress tests.

In [7]:
stress_bc = bc.copy()
stress_bc["Diagnosis_Copy"] = stress_bc["Diagnosis"]
stress_bc["SyntheticID"] = [f"row-{i:06d}" for i in range(len(stress_bc))]

stress_ai = AIDataFacade(
    target="Diagnosis", id_column="SyntheticID",
    task="classification", config=make_config()
)
stress_ai.load(stress_bc)
try:
    print(stress_ai.check_quality())
except Exception as exc:
    print(type(exc).__name__, str(exc))

QualityReport(issues=(QualityIssue(code='high_cardinality', severity=<Severity.WARNING: 'warning'>, message="Column 'SyntheticID' has 569 distinct values, above the configured threshold of 50.", column='SyntheticID', details={'unique_count': 569, 'unique_ratio': 1.0, 'threshold': 50, 'also_id_like': True}, recommendation='One-hot encoding would add one column per distinct value. Consider grouping rare categories or a different encoding.', requires_review=False), QualityIssue(code='possible_id_like', severity=<Severity.WARNING: 'warning'>, message="Column 'SyntheticID' may be a record identifier: 100.0% of its values are distinct and its name or value structure resembles an identifier. This is a heuristic and needs review.", column='SyntheticID', details={'unique_count': 569, 'unique_ratio': 1.0, 'threshold': 0.95, 'detected_kind': 'categorical'}, recommendation='An identifier used as a feature lets a model memorise rows. If this is an identifier, declare it as the id column rather than

### Debate
A competent expert can detect leakage manually, but only if they remember to inspect for it or build dedicated checks. A library contract turns an analyst habit into repeatable system behavior.

Counterargument: system-level rules can also produce false positives or block legitimate expert workflows. Record both.

# Study 2 — Regression: Wine Quality

UCI lists Wine Quality for both classification and regression. Here the quality score is treated as a quantity, so numeric target meaning must be preserved.

In [8]:
wine_X = DATASETS["wine_quality"]["X"].copy()
wine_y = DATASETS["wine_quality"]["y"].copy()
if isinstance(wine_y, pd.DataFrame):
    wine_y = wine_y.iloc[:, 0]
wine = wine_X.copy()
wine["quality"] = pd.to_numeric(wine_y)

X_train, X_valid, y_train, y_valid = train_test_split(
    wine_X, wine["quality"], test_size=VALIDATION_SIZE, random_state=RANDOM_STATE
)

manual_reg = Pipeline([
    ("prep", manual_preprocessor(X_train, scale_numeric=True)),
    ("model", Ridge()),
])
manual_reg.fit(X_train, y_train)
reg_pred = manual_reg.predict(X_valid)
{
    "mae": mean_absolute_error(y_valid, reg_pred),
    "mse": mean_squared_error(y_valid, reg_pred),
    "rmse": mean_squared_error(y_valid, reg_pred) ** 0.5,
    "r2": r2_score(y_valid, reg_pred),
}

{'mae': 0.5658673737830054,
 'mse': 0.5466702693184713,
 'rmse': 0.7393715367245829,
 'r2': 0.2598027510061207}

In [9]:
ai_reg = AIDataFacade(target="quality", task="regression", config=make_config())
ai_reg.load(wine)
reg_quality = ai_reg.check_quality()
ai_reg.prepare()
reg_comparison = ai_reg.compare_models(
    models=["dummy_regressor", "linear_regression", "ridge_regression", "decision_tree_regressor"]
)
ai_reg.select_model("ridge_regression")
reg_training = ai_reg.train()
reg_evaluation = ai_reg.evaluate()
{
    "quality_verdict": ai_reg.verdict.value,
    "selected_model": reg_training.model_name,
    "metrics": {name: reg_evaluation[name].value for name in ("mae", "mse", "rmse", "r2")},
}

{'quality_verdict': 'review_required',
 'selected_model': 'ridge_regression',
 'metrics': {'mae': 0.5658673737830056,
  'mse': 0.5466702693184713,
  'rmse': 0.7393715367245829,
  'r2': 0.2598027510061208}}

# Study 3 — Clustering: Wholesale Customers

No supervised target is used for fitting. Internal metrics are geometric diagnostics, not proof of business meaning.

In [10]:
wh_X = DATASETS["wholesale"]["X"].copy()
wh_Z = manual_preprocessor(wh_X, scale_numeric=True).fit_transform(wh_X)
km = KMeans(n_clusters=3, random_state=RANDOM_STATE, n_init="auto")
labels = km.fit_predict(wh_Z)
dense = wh_Z.toarray() if hasattr(wh_Z, "toarray") else wh_Z
{
    "silhouette": silhouette_score(wh_Z, labels),
    "davies_bouldin": davies_bouldin_score(dense, labels),
    "calinski_harabasz": calinski_harabasz_score(dense, labels),
}

{'silhouette': 0.43916005773275374,
 'davies_bouldin': 1.0578126093969267,
 'calinski_harabasz': 174.7091225697948}

In [11]:
ai_cluster = AIDataFacade(task="clustering", config=make_config())
ai_cluster.load(wh_X)
ai_cluster.check_quality()
ai_cluster.prepare()
# Clustering requires the model directly; there is no selected-model state or automatic winner.
cluster_result = ai_cluster.cluster("kmeans_clustering", n_clusters=3)
cluster_result

ClusteringResult(model_name='kmeans_clustering', estimator=KMeans(n_clusters=3, n_init=10, random_state=42), preprocessor=<aidatasetkit.preprocessing.builder.FittedPreprocessor object at 0x7f4f83060cb0>, labels=array([0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 1,
       1, 2, 0, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 0, 0, 1, 1, 1, 0, 0,
       0, 0, 0, 2, 0, 0, 1, 1, 0, 0, 1, 1, 2, 0, 1, 1, 0, 2, 0, 0, 1, 2,
       1, 0, 1, 1, 1, 1, 1, 0, 0, 1, 1, 0, 1, 1, 1, 0, 0, 1, 0, 2, 2, 1,
       1, 1, 1, 1, 2, 1, 0, 1, 0, 1, 1, 1, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0,
       1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 1, 1, 1, 0, 0, 1, 0, 1, 0,
       1, 1, 1, 1, 1, 2, 1, 2, 1, 1, 1, 1, 0, 0, 1, 1, 1, 0, 1, 1, 1, 0,
       1, 1, 0, 0, 1, 1, 1, 0, 1, 0, 1, 0, 1, 2, 1, 1, 0, 1, 0, 1, 0, 1,
       1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1

### Debate
A manual expert can rank clusterers by silhouette and choose the top score. AIDatasetKit deliberately avoids a universal default clustering winner.

Strength: scientific caution.  
Cost: less convenience.

# Study 4 — Anomaly Detection: Statlog Shuttle

Published labels are used only after fitting as an external diagnostic benchmark.

In [12]:
sh_X = DATASETS["shuttle"]["X"].copy()
sh_y = DATASETS["shuttle"]["y"].copy()
if isinstance(sh_y, pd.DataFrame):
    sh_y = sh_y.iloc[:, 0]
sh_y = pd.to_numeric(sh_y)

sample_n = min(12000, len(sh_X))
idx = np.random.default_rng(RANDOM_STATE).choice(len(sh_X), sample_n, replace=False)
sh_Xs = sh_X.iloc[idx].reset_index(drop=True)
sh_ys = sh_y.iloc[idx].reset_index(drop=True)

sh_Z = manual_preprocessor(sh_Xs, scale_numeric=False).fit_transform(sh_Xs)
iso = IsolationForest(random_state=RANDOM_STATE, contamination="auto")
iso.fit(sh_Z)
manual_is_anomaly = iso.predict(sh_Z) == -1
benchmark_minority = sh_ys.to_numpy() != 1

{
    "flagged_fraction": float(manual_is_anomaly.mean()),
    "benchmark_minority_fraction": float(benchmark_minority.mean()),
    "precision_vs_minority_benchmark": precision_score(benchmark_minority, manual_is_anomaly, zero_division=0),
    "recall_vs_minority_benchmark": recall_score(benchmark_minority, manual_is_anomaly, zero_division=0),
}

{'flagged_fraction': 0.16241666666666665,
 'benchmark_minority_fraction': 0.21275,
 'precision_vs_minority_benchmark': 0.5936377629553617,
 'recall_vs_minority_benchmark': 0.4531923227575401}

In [13]:
ai_anom = AIDataFacade(task="anomaly_detection", config=make_config())
ai_anom.load(sh_Xs)
ai_anom.check_quality()
ai_anom.prepare()
ai_anom.select_model("isolation_forest")
anomaly_result = ai_anom.detect_anomalies()
anomaly_result

AnomalyResult(model_name='isolation_forest', estimator=IsolationForest(random_state=42), preprocessor=<aidatasetkit.preprocessing.builder.FittedPreprocessor object at 0x7f4f830904a0>, is_anomaly=array([False, False, False, ..., False, False, False], shape=(12000,)), anomaly_score=array([0.41448991, 0.39241147, 0.40737434, ..., 0.44916682, 0.48993339,
       0.38017398], shape=(12000,)), raw_score=array([-0.41448991, -0.39241147, -0.40737434, ..., -0.44916682,
       -0.48993339, -0.38017398], shape=(12000,)), backend_labels=array([1, 1, 1, ..., 1, 1, 1], shape=(12000,)), raw_score_name='score_samples', assignment_policy='estimator.predict labels: backend -1 is anomaly and +1 is normal; the fitted detector-specific offset defines this decision', assignment_threshold=-0.5, preprocessing_profile=PreprocessingProfile(requires_scaling=False, supports_sparse_input=True, handles_missing_values=True), plan_fingerprint='9e84da5243a03ef2a9d76cc36ade9146', feature_names=('Rad Flow', 'Fpv Close', 

### Debate
An anomaly score must not be interpreted as probability of fraud, disease, confidence, or calibrated risk. AIDatasetKit's explicit semantics are a scientific advantage, but the library still cannot explain the real-world cause of an anomaly.

# Study 5 — Dimensionality Reduction: Breast Cancer Wisconsin

In [14]:
pca_prep = manual_preprocessor(bc_X, scale_numeric=True)
bc_Z = pca_prep.fit_transform(bc_X)
bc_Z_dense = bc_Z.toarray() if hasattr(bc_Z, "toarray") else bc_Z
pca = PCA(n_components=2, random_state=RANDOM_STATE)
emb = pca.fit_transform(bc_Z_dense)
{
    "shape": emb.shape,
    "explained_variance_ratio": pca.explained_variance_ratio_.tolist(),
    "cumulative_explained_variance": float(pca.explained_variance_ratio_.sum()),
}

{'shape': (569, 2),
 'explained_variance_ratio': [0.44272025607526355, 0.1897118204403308],
 'cumulative_explained_variance': 0.6324320765155944}

In [15]:
ai_dr = AIDataFacade(task="dimensionality_reduction", config=make_config())
ai_dr.load(bc_X)
ai_dr.check_quality()
ai_dr.prepare()
ai_dr.select_model("pca")
dr_result = ai_dr.reduce_dimensions()
dr_result

DimensionalityReductionResult(model_name='pca', estimator=PCA(n_components=2, svd_solver='full'), preprocessor=<aidatasetkit.preprocessing.builder.FittedPreprocessor object at 0x7f4f831454c0>, transformed=array([[ 9.19283683,  1.94858307],
       [ 2.3878018 , -3.76817174],
       [ 5.73389628, -1.0751738 ],
       ...,
       [ 1.25617928, -1.90229671],
       [10.37479406,  1.67201011],
       [-5.4752433 , -0.67063679]], shape=(569, 2)), components=array([[ 0.21890244,  0.10372458,  0.22753729,  0.22099499,  0.14258969,
         0.23928535,  0.25840048,  0.26085376,  0.13816696,  0.06436335,
         0.20597878,  0.01742803,  0.21132592,  0.20286964,  0.01453145,
         0.17039345,  0.15358979,  0.1834174 ,  0.04249842,  0.10256832,
         0.22799663,  0.10446933,  0.23663968,  0.22487053,  0.12795256,
         0.21009588,  0.22876753,  0.25088597,  0.12290456,  0.13178394],
       [-0.23385713, -0.05970609, -0.21518136, -0.23107671,  0.18611302,
         0.15189161,  0.06016536

# Controlled adversarial experiments

Use published datasets as the base, then apply documented perturbations:

- exact target leakage,
- ID leakage,
- evaluation-only extreme values,
- invalid infinity / numeric text,
- unknown category,
- task/target contradiction.

For each perturbation report:
1. whether manual workflow catches it,
2. defensive code required,
3. whether AIDatasetKit catches it,
4. false positives/false negatives,
5. error clarity,
6. whether the library blocks, reviews, or proceeds.

This is the strongest part of the research design because it tests **system contracts**, not just model accuracy.

# Candidate strengths to test

1. System-level safety contracts.
2. Capability-driven preprocessing.
3. Inspectable plans, lineage, and evidence.
4. Reproducibility.
5. Lower orchestration burden.
6. Educationally explicit workflow.
7. Scientifically cautious unsupervised semantics.

# Candidate weaknesses / costs to test

1. Runtime overhead.
2. Reduced flexibility compared with raw sklearn composition.
3. Opinionated policies can block unusual expert workflows.
4. No current CV/tuning core.
5. Correctness depends on library task/capability contracts.
6. Internal unsupervised metrics still cannot prove semantic truth.
7. Evidence fingerprint migrations require disciplined versioning.
8. Smaller exposed algorithm surface than the wider ecosystem.
9. Engineering test success is not production/domain certification.

# Suggested research framing

## Working title
**Safety, Transparency, and Workflow Abstraction in Tabular Machine Learning: A Comparative Evaluation of AIDatasetKit and Conventional scikit-learn Pipelines**

A stronger question than "Is AIDatasetKit better than scikit-learn?" is:

> **What changes when recurrent analytical responsibilities are moved from analyst-authored orchestration into explicit library contracts?**

This creates a real research debate:
- automation vs control,
- safety vs flexibility,
- reproducibility vs overhead,
- educational simplicity vs abstraction,
- system safeguards vs expert judgment.

# Threats to validity

- author bias,
- weak or unfair manual baseline,
- limited dataset representativeness,
- unsupervised metric comparability,
- implementation/version dependence,
- default-policy effects,
- synthetic perturbations layered onto real data,
- external validity.

A serious paper must report results that contradict the expected advantages as clearly as results that support them.

# Controlled adversarial experiments

**This defect was intentionally introduced for the experiment. It is not a defect claimed to exist in the original public dataset.** The next cell constructs the controlled cases; run them separately against the manual and AIDatasetKit paths and populate the safety table with measured outcomes.

In [16]:
# Controlled perturbations: intentionally introduced experiment inputs, never claims about source data.
def controlled_stress_cases(frame: pd.DataFrame, target: str) -> dict[str, pd.DataFrame]:
    base = frame.copy()
    cases = {
        "exact_target_leakage": base.assign(**{f"{target}_Copy": base[target]}),
        "identifier_leakage": base.assign(SyntheticID=[f"row-{i:06d}" for i in range(len(base))]),
        "infinity": base.copy(),
        "numeric_text_corruption": base.copy(),
        "unknown_category": base.copy(),
    }
    numeric = list(base.drop(columns=[target]).select_dtypes(include=[np.number]).columns)
    if numeric:
        cases["infinity"].loc[cases["infinity"].index[0], numeric[0]] = np.inf
        cases["numeric_text_corruption"][numeric[0]] = cases["numeric_text_corruption"][numeric[0]].astype(str)
    categorical = list(base.drop(columns=[target]).select_dtypes(exclude=[np.number]).columns)
    if categorical:
        cases["unknown_category"].loc[cases["unknown_category"].index[0], categorical[0]] = "__held_out_category__"
    return cases

# The comparison report records manual detection, extra manual code, AIDatasetKit outcome,
# error/actionability, and false-positive/false-negative observations for every case.
controlled_stress_cases(bc, "Diagnosis").keys()

dict_keys(['exact_target_leakage', 'identifier_leakage', 'infinity', 'numeric_text_corruption', 'unknown_category'])

# Interpretation, trade-offs, and reporting tables

## Safety outcome table

| Experiment | Manual detected? | Extra manual code | AIDatasetKit detected? | Outcome | Notes |
|---|---:|---|---:|---|---|
| Exact target leakage | To measure | Explicit inspection/check | To measure | Record block/review/proceed | Intentionally introduced; not source-data claim. |
| ID leakage | To measure | Explicit heuristic or schema rule | To measure | Record exclusion/review/proceed | Record false positives as well. |
| Evaluation leakage | To measure | Pipeline and training-only fit discipline | To measure | Compare learned state | Evaluation rows must not affect fitted preprocessing. |
| Infinity / numeric text | To measure | Explicit validation and conversion policy | To measure | Record actionability | Do not call a backend/model error a data-quality diagnosis without evidence. |
| Unknown category | To measure | State encoder policy | To measure | Record external-transform behavior | Keep the policy reproducible. |
| Task contradiction | To measure | Explicit application guard | To measure | Record contract error | No target for clustering, anomaly detection, or reduction. |

## Research comparison table

| Dimension | Conventional expert workflow | AIDatasetKit | Interpretation |
|---|---|---|---|
| Predictive parity | Explicit matched pipeline | Matched registered model and preprocessing plan | Compare only matched rows/configuration/metric. |
| Preprocessing control | Maximum bespoke composition | Capability-driven, documented policy | Control and guardrails are different values. |
| Safety | Expert-built checks | Explicit project contracts | Both may miss cases or create false positives. |
| Reproducibility | Analyst must retain settings and split | Results/plans/split metadata are inspectable | Metadata does not prove scientific truth. |
| Lineage and evidence | Manually assembled | Public result/evidence contracts where available | Evidence excludes runtime estimators and raw rows. |
| Code burden | More analyst orchestration | Smaller guided surface | Fewer lines are not a quality score. |
| Runtime | Often less orchestration work | Includes checks, planning, lineage, and metadata | Measure end-to-end tasks honestly. |
| Flexibility | Stronger for custom research workflows | Deliberately narrower current core | No CV/tuning/AutoML is a current product boundary. |

## When the conventional expert workflow is better

Manual composition is often the better choice for unusual preprocessing, custom estimators or metrics, custom feature engineering, research prototypes, cross-validation experiments, tuning, and uncommon scientific workflows. It is a legitimate advanced interface, not a failure mode.

## When AIDatasetKit may help a learner

AIDatasetKit can make workflow stages, safety findings, preprocessing plans, lineage, and results inspectable. Students should still learn pandas, scikit-learn, metrics, preprocessing, leakage, and statistics. The package should simplify composition, not replace understanding.

## Threats to validity

Author bias, a weak manual baseline, limited dataset representativeness, domain validity, synthetic-perturbation realism, metric comparability, library and scikit-learn version dependence, policy/default dependence, learning-effect bias, and external validity all limit conclusions. A favorable or unfavorable result is reportable; an insufficient experiment is also a valid result.

## References

[1] [UCI Breast Cancer Wisconsin (Diagnostic)](https://archive.ics.uci.edu/dataset/17/breast+cancer+wisconsin+diagnostic)

[2] [UCI Wine Quality](https://archive.ics.uci.edu/dataset/186/wine+quality)

[3] [UCI Wholesale customers](https://archive.ics.uci.edu/dataset/292/wholesale+customers)

[4] [UCI Statlog (Shuttle)](https://archive.ics.uci.edu/dataset/148/statlog+shuttle)

[5] [Creative Commons Attribution 4.0 International](https://creativecommons.org/licenses/by/4.0/)


# Controlled adversarial experiments

**This defect was intentionally introduced for the experiment. It is not a defect claimed to exist in the original public dataset.** The next cell constructs the controlled cases; run them separately against the manual and AIDatasetKit paths and populate the safety table with measured outcomes.

In [17]:
# Controlled perturbations: intentionally introduced experiment inputs, never claims about source data.
def controlled_stress_cases(frame: pd.DataFrame, target: str) -> dict[str, pd.DataFrame]:
    base = frame.copy()
    cases = {
        "exact_target_leakage": base.assign(**{f"{target}_Copy": base[target]}),
        "identifier_leakage": base.assign(SyntheticID=[f"row-{i:06d}" for i in range(len(base))]),
        "infinity": base.copy(),
        "numeric_text_corruption": base.copy(),
        "unknown_category": base.copy(),
    }
    numeric = list(base.drop(columns=[target]).select_dtypes(include=[np.number]).columns)
    if numeric:
        cases["infinity"].loc[cases["infinity"].index[0], numeric[0]] = np.inf
        cases["numeric_text_corruption"][numeric[0]] = cases["numeric_text_corruption"][numeric[0]].astype(str)
    categorical = list(base.drop(columns=[target]).select_dtypes(exclude=[np.number]).columns)
    if categorical:
        cases["unknown_category"].loc[cases["unknown_category"].index[0], categorical[0]] = "__held_out_category__"
    return cases

# The comparison report records manual detection, extra manual code, AIDatasetKit outcome,
# error/actionability, and false-positive/false-negative observations for every case.
controlled_stress_cases(bc, "Diagnosis").keys()

dict_keys(['exact_target_leakage', 'identifier_leakage', 'infinity', 'numeric_text_corruption', 'unknown_category'])

# Interpretation, trade-offs, and reporting tables

## Safety outcome table

| Experiment | Manual detected? | Extra manual code | AIDatasetKit detected? | Outcome | Notes |
|---|---:|---|---:|---|---|
| Exact target leakage | To measure | Explicit inspection/check | To measure | Record block/review/proceed | Intentionally introduced; not source-data claim. |
| ID leakage | To measure | Explicit heuristic or schema rule | To measure | Record exclusion/review/proceed | Record false positives as well. |
| Evaluation leakage | To measure | Pipeline and training-only fit discipline | To measure | Compare learned state | Evaluation rows must not affect fitted preprocessing. |
| Infinity / numeric text | To measure | Explicit validation and conversion policy | To measure | Record actionability | Do not call a backend/model error a data-quality diagnosis without evidence. |
| Unknown category | To measure | State encoder policy | To measure | Record external-transform behavior | Keep the policy reproducible. |
| Task contradiction | To measure | Explicit application guard | To measure | Record contract error | No target for clustering, anomaly detection, or reduction. |

## Research comparison table

| Dimension | Conventional expert workflow | AIDatasetKit | Interpretation |
|---|---|---|---|
| Predictive parity | Explicit matched pipeline | Matched registered model and preprocessing plan | Compare only matched rows/configuration/metric. |
| Preprocessing control | Maximum bespoke composition | Capability-driven, documented policy | Control and guardrails are different values. |
| Safety | Expert-built checks | Explicit project contracts | Both may miss cases or create false positives. |
| Reproducibility | Analyst must retain settings and split | Results/plans/split metadata are inspectable | Metadata does not prove scientific truth. |
| Lineage and evidence | Manually assembled | Public result/evidence contracts where available | Evidence excludes runtime estimators and raw rows. |
| Code burden | More analyst orchestration | Smaller guided surface | Fewer lines are not a quality score. |
| Runtime | Often less orchestration work | Includes checks, planning, lineage, and metadata | Measure end-to-end tasks honestly. |
| Flexibility | Stronger for custom research workflows | Deliberately narrower current core | No CV/tuning/AutoML is a current product boundary. |

## When the conventional expert workflow is better

Manual composition is often the better choice for unusual preprocessing, custom estimators or metrics, custom feature engineering, research prototypes, cross-validation experiments, tuning, and uncommon scientific workflows. It is a legitimate advanced interface, not a failure mode.

## When AIDatasetKit may help a learner

AIDatasetKit can make workflow stages, safety findings, preprocessing plans, lineage, and results inspectable. Students should still learn pandas, scikit-learn, metrics, preprocessing, leakage, and statistics. The package should simplify composition, not replace understanding.

## Threats to validity

Author bias, a weak manual baseline, limited dataset representativeness, domain validity, synthetic-perturbation realism, metric comparability, library and scikit-learn version dependence, policy/default dependence, learning-effect bias, and external validity all limit conclusions. A favorable or unfavorable result is reportable; an insufficient experiment is also a valid result.

## References

[1] [UCI Breast Cancer Wisconsin (Diagnostic)](https://archive.ics.uci.edu/dataset/17/breast+cancer+wisconsin+diagnostic)

[2] [UCI Wine Quality](https://archive.ics.uci.edu/dataset/186/wine+quality)

[3] [UCI Wholesale customers](https://archive.ics.uci.edu/dataset/292/wholesale+customers)

[4] [UCI Statlog (Shuttle)](https://archive.ics.uci.edu/dataset/148/statlog+shuttle)

[5] [Creative Commons Attribution 4.0 International](https://creativecommons.org/licenses/by/4.0/)
